# Conclusions & Next Steps

Cross-notebook summary of `01_eda.ipynb`-`04_anomaly_refund_psp_beta.ipynb`.

## 11. Conclusions

Four distinct anomaly types were identified, each with a different underlying nature:

| Anomaly | Scope | Signature | Likely nature |
|---|---|---|---|
| `bank_id == 777` | 635 rows, 1 day (1 Dec) | Constant 5s latency, 100% fail, ID format anomaly, positioned at the end of the file | Synthetically inserted block |
| psp_gamma latency incident | 5,433 rows, 6 days (12-17 May) | Latency 1-2h, gradual degrade/recover shape, scattered order_id, normal fail rate | Real provider-side operational outage |
| psp_beta refund anomaly | 2,691 rows, 5 days (5-9 Aug) | Flat +10 over-refund in every currency, some on failed transactions, 100% of the year's cases fall in this window | Synthetically inserted / corrupted financial data |
| Refund on failed transaction (`09_anomaly_fail_refund_ratio.ipynb`) | 1,299 rows, all year, all PSPs | 94% exact 0.5x refund ratio on a failed (uncharged) transaction, `applepay`-skewed, 66 rows overlap the psp_beta window via a different mechanism | Synthetically inserted, not time-bound |

A systemic, long-term difference (`psp_alpha`'s persistently higher fail rate) was deliberately **not** treated as an anomaly signal - it spans roughly a third of the dataset and conflicts with the high-class-imbalance premise of the task; it is kept as background context only.

**Methodological lessons carried through this analysis:**
- `describe()` and `value_counts()` on the full dataset hide small clusters (hundreds of rows within a million) - hidden clusters require visualizing distribution shape (histograms) or scanning category x time with `pivot_table` + heatmap.
- Every surprising finding must be verified with an explicit filter and exact numbers before being treated as real - two "new" leads in this analysis (a November fail spike, a currency-driven refund gap) turned out to be a misread axis and a mixed-currency aggregation bug, respectively.
- Aggregating monetary values across rows with different currencies is invalid unless currency is held constant or converted first; row-level comparisons within a single transaction are unaffected by this.

## 12. Extended integrity & signal checks (`01_eda.ipynb` §6, `06_anomaly_invariant_checks.ipynb`)

A systematic pass of cross-column invariants and soft signals, beyond the three clusters above, to check whether they were an exhaustive list or just the most obvious leads.

| Check | Result | Verdict |
|---|---|---|
| Timestamp ordering, `has_refund` consistency, `order_payment_type`/`error_code` exact re-check | 0 exceptions | Confirmed clean |
| Over-refund bound (`refunded_amount > amount`), full year | 2,691 rows, all inside the known 5-9 Aug window | Confirms the psp_beta cluster is exhaustive |
| `ip_country` vs `bin_country` mismatch | 14.43% baseline; explained by customer geography (NA vs rest), no time/category concentration | Background characteristic, not a signal |
| Zero-amount transactions | 0 occurrences | Closed |
| `is_secured` vs fail rate / `error_code` mix | No effect on fail rate; error-code skew consistent with real 3DS decline semantics | Explainable, not a signal |

**Takeaway:** no fourth anomaly cluster surfaced. This is itself evidence (not proof) that the three clusters found in `01`-`04` are the dataset's main anomalies rather than a partial list.

## 13. Amount outliers by category (`07_anomaly_amount_outliers.ipynb`)

Boxplot-per-category scan of `amount` (z-scored within currency to avoid mixing 7 different currency scales), across `payment_method`, `psp_id`, and `bin_country`.

| Check | Result | Verdict |
|---|---|---|
| Currency mix per category | `payment_method`/`psp_id` balanced across currencies; `bin_country` is a near-proxy for currency (70-98% one currency each) | Confound identified and corrected for (z-score within currency) |
| Amount by `payment_method` / `psp_id` | "Outlier" points are a fixed, global recurring-billing price ladder (7-10 tiers per currency, e.g. USD 15/20/30/40/50/60/80/120/150/200), 100% `order_type == recurring` | Legitimate pricing catalog, not a signal |
| Amount by `bin_country` | Initial IQR split looked country-grouped but was a discreteness artifact (`amount` has only ~10 fixed values for `recurring`); categorical tier-mix check (crosstab) shows all 8 countries, including `UKR`, within hundredths of a percentage point on every tier | No country-level pattern |

**Takeaway:** a fifth "false lead" ruled out - what looked like a repeated, suspicious outlier pattern across three independent groupings turned out to be a legitimate, currency-converted subscription price list. Reinforces the same lesson as `psp_alpha` and the `ip`/`bin_country` mismatch: statistically unusual and explainable are not the same as anomalous.

## Next steps

1. ~~Amount distributions by category~~ - done in `07_anomaly_amount_outliers.ipynb`, no new cluster.
2. ~~Refund on failed transaction~~ - done in `09_anomaly_fail_refund_ratio.ipynb`, fourth confirmed cluster.
3. **Isolation Forest / LOF** (see `08_ml_anomaly_scoring.ipynb`) - unsupervised model on engineered features (latency, per-currency amount z-score, `is_fail`, `is_secured`) to catch residual anomalies the rule-based checks don't cover. Sanity-check the model against all four known clusters before trusting it on the rest of the data.
4. **Duplicate-charge check** - `user_id` + `amount` + `created_at` (same-minute) duplicates, a different failure mode than the four known clusters.
5. **Sub-second timestamp patterns** - distribution of `created_at.dt.second` / `.microsecond`; artificial roundness would be a fifth synthetic-insertion signature.
6. **User-level velocity** - transaction frequency per `user_id` in short windows, to catch card-testing/burst behavior.
7. **Assemble the submission** - combine the four confirmed rule-based flags with the ML output (step 3) into `is_anomaly`, and generate the `order_id`/`is_anomaly` CSV.